# Move Segmentation

Demonstrates `kinema.segmentation.moves` on real bouldering COM data.

**Contents**
1. Load COM from Parquet (written by notebook 01)
2. Speed signal — derivation and smoothing
3. Threshold and active-frame mask
4. Detected moves — table and timeline
5. Per-move stats — duration, peak speed, normalized jerk
6. Write `moves.csv`
7. Synthetic sanity checks — 3-move cycle, no-movement, gap merging

**Algorithm** (intentionally crude for MVP — iterating after real footage):
1. Differentiate smoothed COM → speed magnitude
2. Smooth speed with 0.5 s window (suppresses quantization noise)
3. Threshold at `threshold_quantile` of the signal (default 30th percentile)
4. Maximal above-threshold runs → candidate moves
5. Drop moves shorter than `min_move_duration_sec`
6. Merge consecutive moves separated by rest gap shorter than `min_rest_duration_sec`

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from kinema.io.com import read_com
from kinema.segmentation.moves import segment_moves, write_moves, Move
from kinema.kinematics.smoothing import savgol_smooth

DATA_DIR = Path('../data/processed')
COM_PATH = DATA_DIR / 'VID-20260528-WA0044_com.parquet'
FPS = 30.0

print(f'COM file : {COM_PATH}')

## 1. Load COM

In [ ]:
com = read_com(COM_PATH)
com = com.dropna(subset=['com_x', 'com_y', 'com_z']).reset_index(drop=True)

t = com['timestamp_ms'].to_numpy() / 1000.0
duration = t[-1] - t[0]

print(f'Frames : {len(com)}')
print(f'Duration : {duration:.2f} s')
com.head()

## 2. Speed signal

COM speed = ‖∇(com_x, com_y, com_z)‖, computed via `numpy.gradient` (central differences).
A second smoothing pass (0.5 s window) suppresses noise before thresholding.

In [ ]:
dt = 1.0 / FPS
vx = np.gradient(com['com_x'].to_numpy(dtype=np.float64), dt)
vy = np.gradient(com['com_y'].to_numpy(dtype=np.float64), dt)
vz = np.gradient(com['com_z'].to_numpy(dtype=np.float64), dt)
speed_raw = np.sqrt(vx**2 + vy**2 + vz**2)
speed_smooth = savgol_smooth(speed_raw, FPS, window_sec=0.5)

THRESHOLD_QUANTILE = 0.3
threshold = float(np.nanquantile(speed_smooth, THRESHOLD_QUANTILE))

print(f'Speed  min / mean / max : {speed_smooth.min():.4f} / {speed_smooth.mean():.4f} / {speed_smooth.max():.4f}')
print(f'Threshold (Q{THRESHOLD_QUANTILE:.0%})       : {threshold:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3.5))

ax.plot(t, speed_raw,    color='lightsteelblue', linewidth=0.8, label='raw speed', alpha=0.7)
ax.plot(t, speed_smooth, color='steelblue',      linewidth=1.4, label='smoothed speed (0.5 s)')
ax.axhline(threshold, color='tomato', linestyle='--', linewidth=1.1,
           label=f'threshold (Q{THRESHOLD_QUANTILE:.0%} = {threshold:.4f})')
ax.fill_between(t, 0, speed_smooth, where=speed_smooth > threshold,
                color='tomato', alpha=0.18, label='above threshold')

ax.set_xlabel('Time (s)')
ax.set_ylabel('Speed (normalized units/s)')
ax.set_title('COM Speed Signal')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 3. Threshold and active-frame mask

In [ ]:
active = speed_smooth > threshold
pct_active = active.mean() * 100

print(f'Active frames : {active.sum()} / {len(active)}  ({pct_active:.1f}%)')
print(f'Rest   frames : {(~active).sum()} ({100-pct_active:.1f}%)')

## 4. Detected moves

In [ ]:
moves = segment_moves(
    com, FPS,
    threshold_quantile=THRESHOLD_QUANTILE,
    min_move_duration_sec=0.2,
    min_rest_duration_sec=0.15,
)

print(f'Moves detected : {len(moves)}')
print()

rows = [
    {
        'index':          m.index,
        'start_time_sec': f'{m.start_time_sec:.2f}',
        'end_time_sec':   f'{m.end_time_sec:.2f}',
        'duration_sec':   f'{m.duration_sec:.2f}',
        'peak_speed':     f'{m.peak_speed:.4f}',
        'mean_speed':     f'{m.mean_speed:.4f}',
        'normalized_jerk': f'{m.normalized_jerk:.2f}',
    }
    for m in moves
]
pd.DataFrame(rows).set_index('index')

In [ ]:
MOVE_COLORS = plt.cm.tab10.colors

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# ── speed + move spans ──────────────────────────────────────────────────────
ax = axes[0]
ax.plot(t, speed_smooth, color='steelblue', linewidth=1.2, label='smoothed speed')
ax.axhline(threshold, color='tomato', linestyle='--', linewidth=0.9,
           label=f'threshold = {threshold:.4f}')
for m in moves:
    color = MOVE_COLORS[(m.index - 1) % len(MOVE_COLORS)]
    ax.axvspan(m.start_time_sec, m.end_time_sec, alpha=0.25, color=color)
    ax.text(
        (m.start_time_sec + m.end_time_sec) / 2, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else speed_smooth.max(),
        f'M{m.index}', ha='center', va='top', fontsize=8, color=color, fontweight='bold'
    )
ax.set_ylabel('Speed (norm. units/s)')
ax.set_title('COM Speed with Detected Move Spans')
ax.legend(fontsize=9)

# ── COM height (y inverted: 1-y puts up at top) ──────────────────────────────
ax = axes[1]
ax.plot(t, 1 - com['com_y'], color='steelblue', linewidth=1.2, label='height (1 - com_y)')
for m in moves:
    color = MOVE_COLORS[(m.index - 1) % len(MOVE_COLORS)]
    ax.axvspan(m.start_time_sec, m.end_time_sec, alpha=0.25, color=color)
ax.set_ylabel('Height (1 - com_y)')
ax.set_title('COM Vertical Position')
ax.legend(fontsize=9)

# ── COM x ────────────────────────────────────────────────────────────────────
ax = axes[2]
ax.plot(t, com['com_x'], color='steelblue', linewidth=1.2, label='com_x')
for m in moves:
    color = MOVE_COLORS[(m.index - 1) % len(MOVE_COLORS)]
    ax.axvspan(m.start_time_sec, m.end_time_sec, alpha=0.25, color=color)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Horizontal (com_x)')
ax.set_title('COM Horizontal Position')
ax.legend(fontsize=9)

patches = [
    mpatches.Patch(color=MOVE_COLORS[(m.index-1) % len(MOVE_COLORS)], alpha=0.5, label=f'Move {m.index} ({m.duration_sec:.2f}s)')
    for m in moves
]
fig.legend(handles=patches, loc='lower center', ncol=min(len(moves), 6), fontsize=9, bbox_to_anchor=(0.5, -0.02))

plt.suptitle('Detected Moves on COM Trajectory', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Per-move stats

In [ ]:
if not moves:
    print('No moves detected — skipping per-move plots.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    indices = [m.index for m in moves]
    colors  = [MOVE_COLORS[(m.index - 1) % len(MOVE_COLORS)] for m in moves]

    for ax, (attr, ylabel, title) in zip(axes, [
        ('duration_sec',    'Duration (s)',                   'Move Duration'),
        ('peak_speed',      'Peak speed (norm. units/s)',      'Peak Speed per Move'),
        ('normalized_jerk', 'Normalized jerk (lower=smoother)', 'Smoothness (NJ) per Move'),
    ]):
        values = [getattr(m, attr) for m in moves]
        bars = ax.bar(indices, values, color=colors, alpha=0.85, edgecolor='white')
        for bar, val in zip(bars, values):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                val * 1.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8
            )
        ax.set_xlabel('Move index')
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.set_xticks(indices)

    plt.suptitle('Per-Move Statistics', fontweight='bold')
    plt.tight_layout()
    plt.show()

## 6. Write `moves.csv`

In [ ]:
MOVES_PATH = DATA_DIR / 'VID-20260528-WA0044_moves.csv'
write_moves(moves, MOVES_PATH)
print(f'Written : {MOVES_PATH}')

# round-trip verify
df_check = pd.read_csv(MOVES_PATH)
assert len(df_check) == len(moves)
print(f'Rows    : {len(df_check)}')
df_check

## 7. Synthetic sanity checks

Validate the algorithm on known inputs without needing real footage.

In [ ]:
def _synth_com(
    total_sec: float,
    fps: float,
    move_intervals: list[tuple[float, float]],
    amplitude: float = 0.05,
) -> pd.DataFrame:
    """Build COM with sinusoidal motion during move_intervals, flat elsewhere."""
    n = int(total_sec * fps)
    t_s = np.arange(n, dtype=np.float64) / fps
    x = np.zeros(n, dtype=np.float64)
    for start, end in move_intervals:
        mask = (t_s >= start) & (t_s < end)
        dur = end - start
        x[mask] = amplitude * np.sin(2 * np.pi * 3.0 * (t_s[mask] - start) / dur)
    frames = np.arange(n, dtype=np.int32)
    return pd.DataFrame({
        'frame_idx':    frames,
        'timestamp_ms': (frames * 1000.0 / fps).astype(np.int64),
        'com_x': x.astype(np.float32),
        'com_y': np.zeros(n, dtype=np.float32),
        'com_z': np.zeros(n, dtype=np.float32),
    })


FPS_SYNTH = 30.0

# --- 3 clear moves ---
com_3 = _synth_com(12.0, FPS_SYNTH, [(2.0, 3.0), (5.0, 6.0), (8.0, 9.0)])
moves_3 = segment_moves(com_3, FPS_SYNTH)
print(f'3-move trajectory → detected {len(moves_3)} moves  (expected 3)')
assert len(moves_3) == 3, f'Expected 3, got {len(moves_3)}'

# --- no movement ---
n_still = int(FPS_SYNTH * 5)
frames_still = np.arange(n_still, dtype=np.int32)
com_still = pd.DataFrame({
    'frame_idx':    frames_still,
    'timestamp_ms': (frames_still * 1000.0 / FPS_SYNTH).astype(np.int64),
    'com_x': np.zeros(n_still, dtype=np.float32),
    'com_y': np.zeros(n_still, dtype=np.float32),
    'com_z': np.zeros(n_still, dtype=np.float32),
})
moves_still = segment_moves(com_still, FPS_SYNTH)
print(f'Stationary trajectory → detected {len(moves_still)} moves  (expected 0)')
assert len(moves_still) == 0

# --- gap merging: two moves 0.05 s apart → merged to 1 ---
com_gap = _synth_com(10.0, FPS_SYNTH, [(2.0, 3.0), (3.05, 4.05)])
moves_gap = segment_moves(com_gap, FPS_SYNTH, min_rest_duration_sec=0.15)
print(f'Gap-merge trajectory  → detected {len(moves_gap)} move(s)  (expected 1)')
assert len(moves_gap) == 1

# --- sub-duration filter: 0.1 s impulse filtered at 0.8 s minimum ---
com_short = _synth_com(14.0, FPS_SYNTH, [(1.0, 3.0), (8.0, 8.1)])
moves_short = segment_moves(com_short, FPS_SYNTH, min_move_duration_sec=0.8)
print(f'Sub-duration filter   → detected {len(moves_short)} move(s)  (expected 1)')
assert len(moves_short) == 1

print()
print('All assertions passed.')

In [ ]:
# Visualise the 3-move synthetic case
t3 = com_3['timestamp_ms'].to_numpy() / 1000.0
vx3 = np.gradient(com_3['com_x'].to_numpy(dtype=np.float64), 1.0 / FPS_SYNTH)
vy3 = np.gradient(com_3['com_y'].to_numpy(dtype=np.float64), 1.0 / FPS_SYNTH)
vz3 = np.gradient(com_3['com_z'].to_numpy(dtype=np.float64), 1.0 / FPS_SYNTH)
speed3 = savgol_smooth(np.sqrt(vx3**2 + vy3**2 + vz3**2), FPS_SYNTH, window_sec=0.5)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax = axes[0]
ax.plot(t3, com_3['com_x'], color='steelblue', linewidth=1.2)
for m in moves_3:
    color = MOVE_COLORS[(m.index - 1) % len(MOVE_COLORS)]
    ax.axvspan(m.start_time_sec, m.end_time_sec, alpha=0.3, color=color,
               label=f'Move {m.index} ({m.duration_sec:.2f} s)')
ax.set_ylabel('com_x')
ax.set_title('Synthetic 3-Move Trajectory')
ax.legend(fontsize=9)

ax = axes[1]
ax.plot(t3, speed3, color='steelblue', linewidth=1.2)
thr3 = float(np.nanquantile(speed3, 0.3))
ax.axhline(thr3, color='tomato', linestyle='--', linewidth=0.9, label=f'threshold = {thr3:.4f}')
for m in moves_3:
    color = MOVE_COLORS[(m.index - 1) % len(MOVE_COLORS)]
    ax.axvspan(m.start_time_sec, m.end_time_sec, alpha=0.3, color=color)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Smoothed speed')
ax.set_title('Speed Signal with Move Spans')
ax.legend(fontsize=9)

plt.suptitle('Synthetic Sanity Check — 3-Move Cycle', fontweight='bold')
plt.tight_layout()
plt.show()